# Create Argentina MINCYT Awards (multi-funder)

Creates Argentine research-project awards from the MINCYT
"Proyectos de ciencia, tecnología e innovación" CKAN dataset on
datos.gob.ar (2008–2019, ~19,266 projects). One ingest produces rows
attributed to **three OpenAlex funders** based on each project's
`proyecto_fuente`:

  - `ANPCYT`  → Agencia Nacional de Promoción Científica y
                Tecnológica (a.k.a. Agencia I+D+i, F4320334832)
  - `CONICET` → Consejo Nacional de Investigaciones Científicas y
                Técnicas (F4320321594)
  - `INTA`    → Instituto Nacional de Tecnología Agropecuaria
                (F4320326565)

Other (rare) `proyecto_fuente` values are dropped.

**Prerequisites:**
- Run `scripts/local/argentina_mincyt_to_s3.py` first.

**Data source:** datos.gob.ar CKAN, package id
`mincyt_79e643e7-c75f-4f2b-9e8a-66ca8050a262`
**S3 location:** `s3a://openalex-ingest/awards/argentina_mincyt/argentina_mincyt_projects.parquet`

**Currency:** All amounts are in ARS (Argentine peso). The source
exposes a `moneda_id` field but ARS is the default for ANPCYT/CONICET
research grants. The notebook hardcodes `'ARS'`.

**Source amount fields:**
- `monto_total_solicitado` — amount requested by applicant
- `monto_financiado_adjudicado` — amount funded (the funder's share)
- `monto_total_adjudicado` — total awarded (incl. cost-share)

We use `monto_total_adjudicado` as the canonical `amount` for OpenAlex
because it represents the full project cost the funder authorised.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.argentina_mincyt_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/argentina_mincyt/argentina_mincyt_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.argentina_mincyt_raw;

## Step 1.5: Inspect raw + verify amount/currency

Per the how-to Step 1.6, scan all amount-flavored columns and check
distributions before writing the transformation.

In [ ]:
%sql
DESCRIBE openalex.awards.argentina_mincyt_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.argentina_mincyt_raw LIMIT 5;

In [ ]:
%sql
-- proyecto_fuente distribution: which funders are represented?
SELECT proyecto_fuente, COUNT(*) AS cnt
FROM openalex.awards.argentina_mincyt_raw
GROUP BY proyecto_fuente
ORDER BY cnt DESC;

In [ ]:
%sql
-- Amount sanity check (Spanish: monto_total_adjudicado = total amount awarded)
SELECT
  COUNT(*) AS rows,
  COUNT(monto_total_adjudicado) AS has_total_awarded,
  ROUND(MIN(TRY_CAST(monto_total_adjudicado AS DOUBLE)), 0) AS min_amt,
  ROUND(MAX(TRY_CAST(monto_total_adjudicado AS DOUBLE)), 0) AS max_amt,
  ROUND(AVG(TRY_CAST(monto_total_adjudicado AS DOUBLE)), 0) AS avg_amt,
  ROUND(SUM(TRY_CAST(monto_total_adjudicado AS DOUBLE)), 0) AS total_ars
FROM openalex.awards.argentina_mincyt_raw;

## Step 2: Transform to award schema with per-fuente funder mapping

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.argentina_mincyt_awards
USING delta
AS
WITH fuente_to_funder AS (
    -- proyecto_fuente -> OpenAlex funder_id
    SELECT * FROM (VALUES
        ('ANPCYT',  4320334832),  -- Agencia I+D+i
        ('CONICET', 4320321594),  -- CONICET
        ('INTA',    4320326565)   -- INTA
    ) AS t(proyecto_fuente, funder_id)
),
funders_resolved AS (
    SELECT m.proyecto_fuente, f.funder_id, f.display_name, f.ror_id, f.doi
    FROM fuente_to_funder m
    JOIN openalex.common.funder f USING (funder_id)
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(CAST(r.proyecto_id AS STRING))))) % 9000000000 AS id,
    r.titulo AS display_name,
    r.resumen AS description,
    f.funder_id,
    -- funder_award_id: prefer the source's stable codigo_identificacion when present,
    -- fall back to the surrogate proyecto_id
    COALESCE(r.codigo_identificacion, CAST(r.proyecto_id AS STRING)) AS funder_award_id,
    -- Use the total-awarded field (monto_total_adjudicado), not the requested or funder-share
    TRY_CAST(r.monto_total_adjudicado AS DOUBLE) AS amount,
    'ARS' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    -- Surface fuente as funder_scheme so users can filter by source agency in OpenAlex
    r.proyecto_fuente AS funder_scheme,
    'argentina_mincyt' AS provenance,
    -- Source dates are "yyyy/MM/dd HH:mm:ss.SSS" — try multiple parse paths
    COALESCE(
        TRY_TO_DATE(SUBSTRING(r.fecha_inicio, 1, 10), 'yyyy/MM/dd'),
        TRY_TO_DATE(SUBSTRING(r.fecha_inicio, 1, 10), 'yyyy-MM-dd')
    ) AS start_date,
    COALESCE(
        TRY_TO_DATE(SUBSTRING(r.fecha_finalizacion, 1, 10), 'yyyy/MM/dd'),
        TRY_TO_DATE(SUBSTRING(r.fecha_finalizacion, 1, 10), 'yyyy-MM-dd')
    ) AS end_date,
    YEAR(COALESCE(
        TRY_TO_DATE(SUBSTRING(r.fecha_inicio, 1, 10), 'yyyy/MM/dd'),
        TRY_TO_DATE(SUBSTRING(r.fecha_inicio, 1, 10), 'yyyy-MM-dd')
    )) AS start_year,
    YEAR(COALESCE(
        TRY_TO_DATE(SUBSTRING(r.fecha_finalizacion, 1, 10), 'yyyy/MM/dd'),
        TRY_TO_DATE(SUBSTRING(r.fecha_finalizacion, 1, 10), 'yyyy-MM-dd')
    )) AS end_year,
    -- The MINCYT raw data identifies the project director by gender (sexo_director)
    -- but does not expose given/family name fields. Affiliation is also not present.
    -- We leave lead_investigator empty; downstream enrichment can fill from the
    -- proyecto_participante reference table when ingested.
    struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            CAST(NULL AS STRING) AS name,
            'AR' AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CONCAT(
        'https://datos.gob.ar/dataset/mincyt-proyectos-ciencia-tecnologia-innovacion/archivo/proyectos_',
        CAST(r._source_year AS STRING)
    ) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(CAST(r.proyecto_id AS STRING))))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.argentina_mincyt_raw r
JOIN funders_resolved f ON r.proyecto_fuente = f.proyecto_fuente
WHERE r.proyecto_id IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 43

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'argentina_mincyt' AND priority = 43;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    43 AS priority
FROM openalex.awards.argentina_mincyt_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_argentina_awards FROM openalex.awards.argentina_mincyt_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt
FROM openalex.awards.argentina_mincyt_awards;

In [ ]:
%sql
-- Verify the per-fuente split: should match the raw distribution minus dropped fuentes
SELECT funder.display_name, funder_scheme AS proyecto_fuente, COUNT(*) AS cnt,
       ROUND(SUM(amount), 0) AS total_ars
FROM openalex.awards.argentina_mincyt_awards
GROUP BY funder.display_name, funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 70) AS title, funder.display_name AS funder,
       funder_scheme AS fuente, amount, currency, start_date, end_date
FROM openalex.awards.argentina_mincyt_awards LIMIT 10;

In [ ]:
%sql
-- Year coverage
SELECT start_year, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_ars
FROM openalex.awards.argentina_mincyt_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;